In [1]:
%%capture
!pip install -q "transformers>=4.43.0" datasets accelerate huggingface_hub

In [2]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    import os
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    login(tok); os.environ['HUGGING_FACE_HUB_TOKEN'] = tok
    print("HF login ✓")
except Exception as e:
    print(f"Skipping HF login: {e}")


HF login ✓


In [3]:
import os, gc, json, time, random, warnings, urllib.request
from dataclasses import dataclass, field
from typing import List, Dict, Optional

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB


## ROME Config

ROME edits **one layer** with a **rank-one update** (paper §3.1, Eq.2):

```
W_new = W + Λ (C⁻¹k*)ᵀ
```

where  
- `k*` = averaged MLP input at last subject token (paper Eq.3)  
- `v*` = optimised MLP output that causes model to predict new object (paper Eq.4)  
- `C`  = uncentered covariance of k across Wikipedia text (pre-cached)  
- `Λ`  = (v* − W k*) / (C⁻¹k*)ᵀ k*   (closed-form residual scalar)

Unlike MEMIT, there is **no residual spreading** — one layer does all the work.

In [4]:
@dataclass
class ROMEConfig:
    model_name       : str   = 'meta-llama/Llama-3.2-1B'

    # Single edit layer l* — identified by causal tracing peak (paper §2.2).
    # For LLaMA-3.2-1B (16 layers), mid-range peak is typically layer 8.
    layer            : int   = 8
    mlp_module_tmp   : str   = 'model.layers.{}.mlp.down_proj'   # W_proj
    mlp_fc_tmp       : str   = 'model.layers.{}.mlp.gate_proj'   # W_fc (for k*)
    layer_module_tmp : str   = 'model.layers.{}'

    # k* averaging (paper Eq.3): average over N random prefixes ending with subject
    k_num_prefixes   : int   = 50    # paper uses 50
    k_prefix_len     : int   = 5     # random token prefix length (2–10 in paper)

    # v* optimisation (paper Eq.4)
    v_num_grad_steps : int   = 50    # paper: 25 steps sufficient for rank-one
    v_lr             : float = 5e-2
    v_weight_decay   : float = 0.0
    kl_factor        : float = 0.1250
    clamp_norm_factor: float = 4.0

    # Covariance C = E[k kᵀ] (paper Appendix E.5)
    cov_n_texts      : int   = 1000
    cov_ridge_eps    : float = 1e-2

    # Dataset sizes
    n_counterfact    : int   = 20
    n_muse_forget    : int   = 20
    n_muse_retain    : int   = 20

CFG = ROMEConfig()
print("Config ✓  |  Edit layer:", CFG.layer)


Config ✓  |  Edit layer: 8


In [5]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    dtype=torch.float16,
    device_map={'': 0}
)
model.eval()
GPU0 = next(model.parameters()).device
print(f"Model on {GPU0} | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Model on cuda:0 | VRAM: 2.47 GB


In [6]:
def get_module(name: str):
    m = model
    for p in name.split('.'): m = getattr(m, p)
    return m

def get_last_subject_pos(prompt: str, subject: str) -> int:
    ids = tokenizer(prompt, return_tensors='pt')['input_ids'][0]
    n   = len(ids)
    for toks in [
        tokenizer(subject, add_special_tokens=False)['input_ids'],
        tokenizer(' '+subject, add_special_tokens=False)['input_ids'],
    ]:
        for i in range(n-len(toks), -1, -1):
            if ids[i:i+len(toks)].tolist() == toks:
                return i+len(toks)-1
    # char-level fallback
    enc     = tokenizer(prompt, return_tensors='pt', return_offsets_mapping=True)
    offsets = enc['offset_mapping'][0]
    cp      = prompt.lower().rfind(subject.lower())
    if cp != -1:
        lc = cp+len(subject)-1
        for ti in range(len(offsets)-1,-1,-1):
            s,e = offsets[ti].tolist()
            if s<=lc<=e: return ti
    return n-1

@torch.no_grad()
def token_log_prob(prompt: str, target: str) -> float:
    enc   = tokenizer(prompt+' '+target, return_tensors='pt').to(GPU0)
    p_len = tokenizer(prompt, return_tensors='pt')['input_ids'].shape[1]
    lp    = torch.log_softmax(model(**enc).logits[0], dim=-1)
    tids  = enc['input_ids'][0][p_len:]
    if len(tids)==0: return 0.0
    return sum(lp[p_len-1+i,tid].item() for i,tid in enumerate(tids))/len(tids)

@torch.no_grad()
def top1_token(prompt: str) -> str:
    enc = tokenizer(prompt, return_tensors='pt').to(GPU0)
    return tokenizer.decode([model(**enc).logits[0,-1].argmax().item()]).strip()

print("Helpers ✓")


Helpers ✓


In [7]:
from datasets import load_dataset

# ── CounterFact ───────────────────────────────────────────────────────────────
CF_CACHE = '/kaggle/working/counterfact.json'
if not os.path.exists(CF_CACHE):
    print("Downloading CounterFact ...")
    urllib.request.urlretrieve(
        'https://rome.baulab.info/data/dsets/counterfact.json', CF_CACHE)
with open(CF_CACHE) as f:
    raw_cf = json.load(f)
print(f"CounterFact records: {len(raw_cf)}")

@dataclass
class CFRecord:
    subject           : str
    prompt            : str
    target_new        : str
    ground_truth      : str
    rephrase_prompts  : List[str]
    neighborhood_prompts: List[str]

def parse_cf(raw, n=None) -> List[CFRecord]:
    out = []
    for item in (raw[:n] if n else raw):
        req  = item.get('requested_rewrite',{})
        subj = req.get('subject','')
        tt   = req.get('target_true',{})
        tn   = req.get('target_new',{})
        out.append(CFRecord(
            subject              = subj,
            prompt               = req.get('prompt','{}').format(subj),
            target_new           = tn.get('str','') if isinstance(tn,dict) else str(tn),
            ground_truth         = tt.get('str','') if isinstance(tt,dict) else str(tt),
            rephrase_prompts     = item.get('paraphrase_prompts',[]),
            neighborhood_prompts = item.get('neighborhood_prompts',[]),
        ))
    return out

cf_records = parse_cf(raw_cf, n=CFG.n_counterfact)
print(f"Using {len(cf_records)} CounterFact records")
print(f'Sample: "{cf_records[0].prompt}" → "{cf_records[0].target_new}"')

CounterFact records: 21919
Using 20 CounterFact records
Sample: "The mother tongue of Danielle Darrieux is" → "English"


In [8]:
# ── MUSE-News knowmem ─────────────────────────────────────────────────────────
print("Loading MUSE-News knowmem ...")
muse_ds = load_dataset("muse-bench/MUSE-News", "knowmem")
print("Splits:", list(muse_ds.keys()))

@dataclass
class MuseRecord:
    id           : str
    subject      : str
    prompt       : str
    target_new   : str
    ground_truth : str
    split        : str
    raw          : dict = field(default_factory=dict)

def extract_subject(text: str) -> str:
    for sep in [' is ',' was ',' has ',' were ',' are ']:
        idx = text.lower().find(sep)
        if idx != -1: return text[:idx].strip()
    return ' '.join(text.split()[:4]).rstrip('.,;:')

def parse_muse(ds_split, split_name: str, n: int=None) -> List[MuseRecord]:
    out  = []
    data = ds_split.select(range(min(n,len(ds_split)))) if n else ds_split
    for i, item in enumerate(data):
        prompt = (item.get('question') or item.get('prompt') or
                  item.get('input')   or item.get('text',''))
        gt     = (item.get('answer')  or item.get('ground_truth') or
                  item.get('target') or item.get('output',''))
        if isinstance(gt,list): gt=gt[0] if gt else ''
        gt = str(gt).strip()
        subj   = item.get('subject') or item.get('entity') or extract_subject(prompt)
        tgt_new= item.get('new_target') or item.get('wrong_answer') or 'unknown'
        out.append(MuseRecord(
            id=str(item.get('id',i)), subject=subj, prompt=prompt,
            target_new=tgt_new, ground_truth=gt, split=split_name, raw=dict(item)))
    return out

forget_split  = muse_ds.get('forget') or list(muse_ds.values())[0]
retain_split  = muse_ds.get('retain') or list(muse_ds.values())[-1]
muse_forget   = parse_muse(forget_split, 'forget', n=CFG.n_muse_forget)
muse_retain   = parse_muse(retain_split, 'retain', n=CFG.n_muse_retain)
print(f"MUSE forget: {len(muse_forget)} | retain: {len(muse_retain)}")
print(f'Sample: "{muse_forget[0].prompt[:70]}"')

Loading MUSE-News knowmem ...
Splits: ['retain_qa_icl', 'retain_qa', 'forget_qa', 'forget_qa_icl']
MUSE forget: 10 | retain: 10
Sample: "Who apologized to Mr. Farage for 'deeply inappropriate' comments after"


## Covariance C (paper Appendix E.5)

`C = E[k kᵀ]` — uncentered covariance of the MLP input `k` at layer `l*`, collected over Wikipedia text.  
Used in the rank-one update: `W_new = W + Λ (C⁻¹k*)ᵀ`  
C⁻¹ down-weights directions the MLP uses constantly, so the edit avoids disrupting other memories.  
**One covariance, one layer** — much simpler than MEMIT which needs one per layer in a range.

In [9]:
_cov_inv: Optional[torch.Tensor] = None

def get_cov_inv() -> torch.Tensor:
    global _cov_inv
    if _cov_inv is not None:
        return _cov_inv

    print(f'[Cov] Layer {CFG.layer} ...', end=' ', flush=True)
    mod  = get_module(CFG.mlp_module_tmp.format(CFG.layer))
    d_in = mod.weight.shape[1]

    buf = []
    def hook(m, inp, out):
        buf.append(inp[0].detach().float().reshape(-1, d_in).cpu())
    h = mod.register_forward_hook(hook)

    # Source 1: MUSE raw split (always available, full articles)
    texts = []
    raw_split = muse_ds.get('raw') or muse_ds.get('train')
    if raw_split:
        for item in raw_split:
            t = (item.get('text') or item.get('content') or
                 item.get('document') or item.get('input',''))
            for i in range(0, min(len(t), 6000), 300):
                texts.append(t[i:i+300])

    # Source 2: Wikipedia streaming (if internet available)
    try:
        from datasets import load_dataset as ld
        wiki = ld('wikipedia','20220301.en', split='train', streaming=True)
        texts += [ex['text'][:300] for ex,_ in zip(wiki, range(CFG.cov_n_texts))]
        print(f'(wiki+muse: {len(texts)} texts)', end=' ')
    except Exception as e:
        print(f'(wiki failed: {type(e).__name__}, using MUSE raw only: {len(texts)} texts)', end=' ')

    # Fallback: wikitext-103
    if len(texts) < 200:
        try:
            from datasets import load_dataset as ld
            wt = ld('wikitext','wikitext-103-raw-v1', split='train')
            texts += [ex['text'] for ex in wt if len(ex['text'])>80][:CFG.cov_n_texts]
            print(f'(wikitext fallback: {len(texts)} texts)', end=' ')
        except Exception:
            pass

    # Last resort: repeat retain prompts
    if len(texts) < 100:
        texts = ([r.prompt for r in muse_retain] +
                 [r.prompt for r in muse_forget]) * 50
        print(f'(MUSE prompts only x50: {len(texts)} texts)', end=' ')

    random.shuffle(texts)

    running = torch.zeros(d_in, d_in)
    total   = 0
    with torch.no_grad():
        for txt in texts:
            if not txt.strip(): continue
            enc = tokenizer(txt, return_tensors='pt',
                            max_length=128, truncation=True).to(GPU0)
            model(**enc)
            if buf:
                k        = buf.pop()
                running += k.T @ k
                total   += k.shape[0]
    h.remove()

    print(f'tokens={total}', end=' ')
    if total < 5000:
        print(f'\n  [WARN] Only {total} tokens — covariance may be unreliable. '
              f'Consider adding more text sources.')

    C     = running / max(total, 1)
    eps   = CFG.cov_ridge_eps * C.diagonal().mean().item()
    C_reg = C + eps * torch.eye(d_in, dtype=C.dtype)
    assert torch.linalg.eigvalsh(C_reg).min().item() > 0,         'C not PD — increase cov_ridge_eps'

    _cov_inv = torch.linalg.inv(C_reg)
    print(f'done  eps={eps:.2e}  d_in={d_in}')
    return _cov_inv

print("Covariance ✓")


Covariance ✓


## Step 1 — Compute k* (paper Eq.3)

`k* = (1/N) Σⱼ k(xⱼ + s)`

where `k(x)` = MLP intermediate activation at last subject token after the nonlinearity inside W_fc.  
Averaging over N=50 random prefixes makes k* robust to context variation.

In [10]:
@torch.no_grad()
def compute_k_star(subject: str) -> torch.Tensor:
    """Average MLP input at last subject token over N random prefixes (paper Eq.3)."""
    mod_fc   = get_module(CFG.mlp_fc_tmp.format(CFG.layer))   # gate_proj → k
    mod_down = get_module(CFG.mlp_module_tmp.format(CFG.layer))
    d_in     = mod_down.weight.shape[1]

    # Generate N random prefixes ending with subject (paper: 2–10 random tokens)
    vocab_size = tokenizer.vocab_size
    k_acc      = torch.zeros(d_in, dtype=torch.float32)
    n_collected = 0

    subj_ids = tokenizer(' '+subject, add_special_tokens=False,
                         return_tensors='pt')['input_ids'].to(GPU0)

    for _ in range(CFG.k_num_prefixes):
        # Random prefix of random length 2–10
        plen   = random.randint(2, CFG.k_prefix_len+5)
        prefix = torch.randint(0, vocab_size, (1, plen), device=GPU0)
        inp    = torch.cat([prefix, subj_ids], dim=1)

        # Subject last token position
        s_pos  = inp.shape[1] - 1

        cap = {}
        def hk(m, inp_, out, c=cap):
            # k(x) = activation after gate_proj nonlinearity
            # In LLaMA: down_proj input = silu(gate_proj) * up_proj
            # We capture down_proj input (= k in paper notation)
            c['k'] = inp_[0].detach().float()
        hdl = mod_down.register_forward_hook(hk)
        try:
            model(inp)
        except Exception:
            hdl.remove(); continue
        hdl.remove()

        if 'k' in cap:
            k_acc    += cap['k'][0, s_pos].cpu()
            n_collected += 1

    return (k_acc / max(n_collected, 1))  # (d_in,)

print("k* computation ✓")


k* computation ✓


## Step 2 — Optimise v* (paper Eq.4)

`v* = argmin_z  (1/N) Σⱼ [ -log P_G(o*|xⱼ+p) + λ_KL · KL(G(xⱼ+p') ‖ G_orig(xⱼ+p')) ]`

- Term (a): cross-entropy loss pushing z to predict the new object o*
- Term (b): KL divergence on the "{subject} is a" prompt to prevent essence drift
- The optimisation runs on `z` — **no weight change yet**. Weights are updated only in Step 3.

In [11]:
def compute_v_star(subject: str, prompt: str, target_new: str) -> torch.Tensor:
    """Optimise v* to predict target_new (paper Eq.4). Returns (d_out,) on CPU."""
    anchor     = CFG.layer
    lm         = get_module(CFG.layer_module_tmp.format(anchor))
    mod_down   = get_module(CFG.mlp_module_tmp.format(anchor))
    d_out      = mod_down.weight.shape[0]

    tgt_ids       = tokenizer(' '+target_new, add_special_tokens=False,
                               return_tensors='pt')['input_ids'].to(GPU0)
    pmt_ids_plain = tokenizer(prompt, return_tensors='pt')['input_ids'].to(GPU0)
    s_pos_plain   = get_last_subject_pos(prompt, subject)

    # Essence prompt p' = "{subject} is a" (paper §3, Step 2)
    essence_prompt = f'{subject} is a'
    essence_ids    = tokenizer(essence_prompt, return_tensors='pt')['input_ids'].to(GPU0)

    # Baseline hidden state h0 at anchor layer
    cap = {}
    def snap(m, inp, out, c=cap):
        c['h'] = (out[0] if isinstance(out,tuple) else out).detach()
    hdl = lm.register_forward_hook(snap)
    with torch.no_grad():
        full_plain = torch.cat([pmt_ids_plain, tgt_ids], dim=1)
        model(full_plain)
    hdl.remove()
    h0    = cap['h'][0, s_pos_plain].float().clone()
    delta = torch.zeros_like(h0, requires_grad=True)

    opt = torch.optim.Adam([delta], lr=CFG.v_lr, weight_decay=CFG.v_weight_decay)

    # Random prefixes for v* (same as k* — paper uses same 50 prefixes)
    vocab_size = tokenizer.vocab_size
    prefixes   = [torch.randint(0, vocab_size,
                                (1, random.randint(2, CFG.k_prefix_len+5)),
                                device=GPU0)
                  for _ in range(min(10, CFG.v_num_grad_steps))]  # 10 prefix samples

    for step in range(CFG.v_num_grad_steps):
        opt.zero_grad()
        total_ce = torch.tensor(0.0, device=GPU0)
        total_kl = torch.tensor(0.0, device=GPU0)

        # Use a rotating prefix each step (paper: average over xⱼ)
        prefix  = prefixes[step % len(prefixes)]
        pmt_v   = torch.cat([prefix,
                              pmt_ids_plain[:,1:] if pmt_ids_plain[0,0]==tokenizer.bos_token_id
                              else pmt_ids_plain], dim=1)
        full_v  = torch.cat([pmt_v, tgt_ids], dim=1)
        off     = pmt_v.shape[1] - pmt_ids_plain.shape[1]
        sp_v    = max(0, s_pos_plain + off)

        def patch_layer(m, inp, out, sp=sp_v, d=delta):
            h  = (out[0] if isinstance(out,tuple) else out).float().clone()
            if sp < h.shape[1]:
                h[0, sp] = h[0, sp] + d
            dt = out[0].dtype if isinstance(out,tuple) else out.dtype
            h  = h.to(dt)
            return (h,)+out[1:] if isinstance(out,tuple) else h

        ph  = lm.register_forward_hook(patch_layer)
        log = model(full_v).logits[0].float()
        ph.remove()

        t0 = pmt_v.shape[1]
        n_tgt = tgt_ids.shape[1]
        if t0-1+n_tgt <= log.shape[0]:
            total_ce = F.cross_entropy(log[t0-1:t0-1+n_tgt], tgt_ids[0])

        # KL on essence prompt (term b)
        ess_v  = torch.cat([prefix, essence_ids[:,1:]
                             if essence_ids[0,0]==tokenizer.bos_token_id
                             else essence_ids], dim=1)
        ess_sp = max(0, sp_v)
        def patch_ess(m, inp, out, sp=ess_sp, d=delta):
            h  = (out[0] if isinstance(out,tuple) else out).float().clone()
            if sp < h.shape[1]:
                h[0, sp] = h[0, sp] + d
            dt = out[0].dtype if isinstance(out,tuple) else out.dtype
            h  = h.to(dt)
            return (h,)+out[1:] if isinstance(out,tuple) else h

        ph2 = lm.register_forward_hook(patch_ess)
        log_ess = model(ess_v).logits[0].float()
        ph2.remove()
        with torch.no_grad():
            ref_ess = model(ess_v).logits[0].float()
        total_kl = CFG.kl_factor * F.kl_div(
            torch.log_softmax(log_ess, dim=-1),
            torch.softmax(ref_ess,    dim=-1),
            reduction='batchmean')

        loss = total_ce + total_kl
        loss.backward()

        with torch.no_grad():
            max_n = CFG.clamp_norm_factor * h0.norm()
            if delta.norm() > max_n:
                delta.data = delta.data * max_n / (delta.norm()+1e-8)

        opt.step()

    v_star = (h0 + delta.detach()).cpu()
    return v_star   # (d_out,) — this IS the optimised MLP output value

print("v* optimiser ✓")


v* optimiser ✓


## Step 3 — Rank-One Weight Update (paper Eq.2)

```
W_new = W + Λ (C⁻¹k*)ᵀ

where  Λ = (v* − W k*) / (C⁻¹k*)ᵀ k*
```

- `v* − Wk*` = residual: what W currently outputs for k* vs what we want
- `(C⁻¹k*)` = covariance-normalised key — steers update away from busy directions  
- The denominator `(C⁻¹k*)ᵀ k*` is a scalar — normalises the magnitude

This is a **rank-one** update (`Λ · (C⁻¹k*)ᵀ` has rank 1).  
It satisfies `W_new k* = v*` exactly, while minimising interference with existing memories.

In [12]:
def apply_rome_update(k_star: torch.Tensor, v_star: torch.Tensor):
    """
    Compute and apply rank-one update to W_proj at CFG.layer (paper Eq.2).
    W_new = W + Λ (C⁻¹k*)ᵀ
    Λ     = (v* - W k*) / (C⁻¹k*)ᵀ k*
    """
    mod = get_module(CFG.mlp_module_tmp.format(CFG.layer))
    dev = mod.weight.device
    W   = mod.weight.float()                # (d_out, d_in)

    k   = k_star.float().to(dev)            # (d_in,)
    v   = v_star.float().to(dev)            # (d_out,)
    Ci  = get_cov_inv().float().to(dev)     # (d_in, d_in)

    Cik    = Ci @ k                         # (d_in,)  — covariance-normalised key
    Wk     = W @ k                          # (d_out,) — current output for k*
    resid  = v - Wk                         # (d_out,) — what we still need to add
    denom  = Cik @ k                        # scalar

    Lambda = resid / (denom + 1e-8)         # (d_out,) — Eq.2 Λ vector
    dW     = torch.outer(Lambda, Cik)       # (d_out, d_in) — rank-one update

    with torch.no_grad():
        mod.weight.add_(dW.to(mod.weight.dtype))

    # Verify: W_new k* ≈ v*
    with torch.no_grad():
        W_new = mod.weight.float()
        err   = (W_new @ k - v).norm().item()
    print(f'    ‖ΔW‖={dW.norm():.4f}  fit_err‖W_new·k*−v*‖={err:.4f}')

print("Rank-one update ✓")


Rank-one update ✓


In [13]:
def apply_rome(subject: str, prompt: str, target_new: str, label: str=''):
    """Full ROME pipeline for one edit (paper §3.1)."""
    print(f'  [ROME] {label}  "{prompt[:55]}" → "{target_new}"')
    k = compute_k_star(subject)
    v = compute_v_star(subject, prompt, target_new)
    apply_rome_update(k, v)
    gc.collect(); torch.cuda.empty_cache()

print("apply_rome ✓")


apply_rome ✓


## Run ROME on CounterFact

CounterFact is ROME's own benchmark (paper §3.3).  
Each edit: insert counterfactual `(subject, relation, new_object)` into the model.  
We store pre-edit weights so we can reset between dataset runs.

In [14]:

# Save original weights before any edits
orig_weight = get_module(CFG.mlp_module_tmp.format(CFG.layer)).weight.data.clone()

def reset_weights():
    """Restore original W_proj to run fresh on next dataset."""
    mod = get_module(CFG.mlp_module_tmp.format(CFG.layer))
    with torch.no_grad():
        mod.weight.copy_(orig_weight)
    gc.collect(); torch.cuda.empty_cache()
    print("Weights reset ✓")

# ── Pre-edit snapshot ─────────────────────────────────────────────────────────
print("=== CounterFact Pre-edit ===")
cf_pre = []
for r in cf_records[:5]:
    lp  = token_log_prob(r.prompt, r.target_new)
    t1  = top1_token(r.prompt)
    cf_pre.append({'lp':lp,'top1':t1})
    print(f'  "{r.prompt[:55]}"  gt="{r.ground_truth}"  top1="{t1}"  lp(new)={lp:.3f}')

# ── ROME edits ────────────────────────────────────────────────────────────────
print("\n=== Applying ROME to CounterFact ===")
t0 = time.time()
for i, r in enumerate(cf_records):
    apply_rome(r.subject, r.prompt, r.target_new, label=f'CF {i+1}/{len(cf_records)}')
print(f"\nDone  {time.time()-t0:.1f}s  ({(time.time()-t0)/len(cf_records):.2f}s/edit)")

# ── Post-edit check ───────────────────────────────────────────────────────────
print("\n=== CounterFact Post-edit (first 5) ===")
for r, pre in zip(cf_records[:5], cf_pre):
    post = token_log_prob(r.prompt, r.target_new)
    ok   = '✓' if post > pre['lp'] else '✗'
    print(f'  {ok} pre={pre["lp"]:.3f}  post={post:.3f}  Δ={post-pre["lp"]:+.3f}  '
          f'top1="{top1_token(r.prompt)}"  "{r.prompt[:45]}"')


=== CounterFact Pre-edit ===
  "The mother tongue of Danielle Darrieux is"  gt="French"  top1="French"  lp(new)=-4.086
  "The official religion of Edwin of Northumbria is"  gt="Christianity"  top1="the"  lp(new)=-5.637
  "Toko Yasuda, the"  gt="guitar"  top1=""  lp(new)=-10.820
  "Autonomous University of Madrid, which is located in"  gt="Spain"  top1="the"  lp(new)=-9.555
  "What is the twin city of Lyon? It is"  gt="Beirut"  top1="the"  lp(new)=-11.055

=== Applying ROME to CounterFact ===
  [ROME] CF 1/20  "The mother tongue of Danielle Darrieux is" → "English"
[Cov] Layer 8 ... (wiki failed: RuntimeError, using MUSE raw only: 0 texts) (wikitext fallback: 1000 texts) tokens=109993 done  eps=2.87e-05  d_in=8192
    ‖ΔW‖=10.8329  fit_err‖W_new·k*−v*‖=0.0006
  [ROME] CF 2/20  "The official religion of Edwin of Northumbria is" → "Islam"
    ‖ΔW‖=9.2108  fit_err‖W_new·k*−v*‖=0.0010
  [ROME] CF 3/20  "Toko Yasuda, the" → "piano"
    ‖ΔW‖=11.0769  fit_err‖W_new·k*−v*‖=0.0008
  [ROME] CF 4/

## CounterFact Evaluation (paper §3.4)

| Metric | Meaning |
|---|---|
| **ES** Efficacy Score | P(new) > P(old) post-edit |
| **PS** Paraphrase Score | ES on rephrased prompts |
| **NS** Neighbourhood Score | Nearby subjects unaffected |
| **S** Harmonic mean | ES × PS × NS combined |

In [15]:
def eval_counterfact(records: List[CFRecord]) -> dict:
    es, ps, ns = [], [], []
    for r in records:
        lp_new = token_log_prob(r.prompt, r.target_new)
        lp_old = token_log_prob(r.prompt, r.ground_truth)
        es.append(int(lp_new > lp_old))

        for rp in r.rephrase_prompts[:3]:
            ps.append(int(token_log_prob(rp, r.target_new) >
                          token_log_prob(rp, r.ground_truth)))

        for np_ in r.neighborhood_prompts[:5]:
            # Specificity: neighbour still prefers the TRUE answer
            ns.append(int(token_log_prob(np_, r.ground_truth) >
                          token_log_prob(np_, r.target_new)))

    ES = float(np.mean(es)) if es else 0.0
    PS = float(np.mean(ps)) if ps else 0.0
    NS = float(np.mean(ns)) if ns else 0.0
    S  = 3/(1/(ES+1e-9)+1/(PS+1e-9)+1/(NS+1e-9))
    return dict(efficacy=ES, paraphrase=PS, neighbourhood=NS, score_HM=S)

print("\n=== CounterFact Metrics ===")
cf_metrics = eval_counterfact(cf_records)
for k,v in cf_metrics.items():
    bar = '█'*int(v*30)
    print(f'  {k:<18} {v:.3f}  {bar}')



=== CounterFact Metrics ===
  efficacy           0.950  ████████████████████████████
  paraphrase         0.875  ██████████████████████████
  neighbourhood      0.570  █████████████████
  score_HM           0.760  ██████████████████████


## Run ROME on MUSE-News (knowmem forget split)

ROME edits one fact at a time — applied sequentially to each forget record.  
After editing, we check:
- **Forget efficacy**: model no longer outputs the ground truth
- **Retain locality**: retain-split facts unaffected

In [16]:
reset_weights()   # fresh start for MUSE run

print("=== MUSE Pre-edit ===")
muse_pre = []
for r in muse_forget[:5]:
    lp  = token_log_prob(r.prompt, r.ground_truth)
    t1  = top1_token(r.prompt)
    muse_pre.append({'lp':lp,'top1':t1})
    print(f'  "{r.prompt[:60]}"  gt="{r.ground_truth[:15]}"  top1="{t1}"')

print("\n=== Applying ROME to MUSE forget split ===")
t0 = time.time()
for i, r in enumerate(muse_forget):
    apply_rome(r.subject, r.prompt, r.target_new,
               label=f'MUSE {i+1}/{len(muse_forget)}')
print(f"\nDone  {time.time()-t0:.1f}s")

print("\n=== MUSE Post-edit (first 5) ===")
for r, pre in zip(muse_forget[:5], muse_pre):
    post = token_log_prob(r.prompt, r.ground_truth)
    ok   = '✓ UN' if post < pre['lp'] else '✗ KP'
    print(f'  {ok}  pre={pre["lp"]:.3f}  post={post:.3f}  Δ={post-pre["lp"]:+.3f}  '
          f'new_top1="{top1_token(r.prompt)}"')


Weights reset ✓
=== MUSE Pre-edit ===
  "Who apologized to Mr. Farage for 'deeply inappropriate' comm"  gt="Dame Alison Ros"  top1="|"
  "What is the main feeling expressed by Hearts interim manager"  gt="Frustration"  top1="("
  "What significant event involving Martin McGuinness and the B"  gt="Martin McGuinne"  top1="The"
  "Which Liverpool player announced on Friday he would end his "  gt="Firmino"  top1="("
  "What condition must users meet before playing the street pia"  gt="disinfect their"  top1="The"

=== Applying ROME to MUSE forget split ===
  [ROME] MUSE 1/10  "Who apologized to Mr. Farage for 'deeply inappropriate'" → "unknown"
    ‖ΔW‖=7.2633  fit_err‖W_new·k*−v*‖=0.0008
  [ROME] MUSE 2/10  "What is the main feeling expressed by Hearts interim ma" → "unknown"
    ‖ΔW‖=7.5341  fit_err‖W_new·k*−v*‖=0.0013
  [ROME] MUSE 3/10  "What significant event involving Martin McGuinness and " → "unknown"
    ‖ΔW‖=6.5562  fit_err‖W_new·k*−v*‖=0.0010
  [ROME] MUSE 4/10  "Which Liverpool

In [17]:
def eval_muse(forget_recs: List[MuseRecord],
              retain_recs: List[MuseRecord]) -> dict:
    fe, flp, rl, ps = [], [], [], []

    for r in forget_recs:
        lp_gt  = token_log_prob(r.prompt, r.ground_truth)
        lp_new = token_log_prob(r.prompt, r.target_new)
        t1     = top1_token(r.prompt)
        fe.append(int(t1.lower() != r.ground_truth.lower()[:len(t1)]))
        flp.append(int(lp_new > lp_gt))
        for pp in (r.raw.get('paraphrase_prompts') or
                   r.raw.get('rephrase_prompts') or [])[:3]:
            ps.append(int(token_log_prob(pp, r.target_new) >
                          token_log_prob(pp, r.ground_truth)))

    for r in retain_recs:
        rl.append(int(token_log_prob(r.prompt, r.ground_truth) >=
                      token_log_prob(r.prompt, r.target_new)))

    FE  = float(np.mean(fe))  if fe  else 0.0
    FLP = float(np.mean(flp)) if flp else 0.0
    RL  = float(np.mean(rl))  if rl  else 0.0
    PS  = float(np.mean(ps))  if ps  else 0.0
    HM  = 2/(1/(FE+1e-9)+1/(RL+1e-9))
    return dict(forget_efficacy=FE, forget_lp_flip=FLP,
                retain_locality=RL, paraphrase=PS, HM_FE_RL=HM)

print("\n=== MUSE Metrics ===")
muse_metrics = eval_muse(muse_forget, muse_retain)
for k,v in muse_metrics.items():
    bar = '█'*int(v*30)
    print(f'  {k:<22} {v:.3f}  {bar}')



=== MUSE Metrics ===
  forget_efficacy        1.000  ██████████████████████████████
  forget_lp_flip         1.000  ██████████████████████████████
  retain_locality        0.300  █████████
  paraphrase             0.000  
  HM_FE_RL               0.462  █████████████


In [18]:
print("\n=== Per-record detail (MUSE forget) ===")
print(f'{"St":<4} {"Prompt":<52} {"GT":<12} {"Top-1":<12} {"lp(gt)":>7} {"lp(new)":>7}')
print('-'*100)
for r in muse_forget:
    lp_gt  = token_log_prob(r.prompt, r.ground_truth)
    lp_new = token_log_prob(r.prompt, r.target_new)
    t1     = top1_token(r.prompt)
    ok     = '✓U' if lp_new>lp_gt else '✗K'
    print(f'{ok:<4} {r.prompt[:51]:<52} {r.ground_truth[:11]:<12} '
          f'{t1[:11]:<12} {lp_gt:>7.3f} {lp_new:>7.3f}')


=== Per-record detail (MUSE forget) ===
St   Prompt                                               GT           Top-1         lp(gt) lp(new)
----------------------------------------------------------------------------------------------------
✓U   Who apologized to Mr. Farage for 'deeply inappropri  Dame Alison  unknown      -10.108  -0.004
✓U   What is the main feeling expressed by Hearts interi  Frustration  unknown       -5.011  -0.280
✓U   What significant event involving Martin McGuinness   Martin McGu  unknown       -4.376  -0.001
✓U   Which Liverpool player announced on Friday he would  Firmino      unknown       -9.455  -0.004
✓U   What condition must users meet before playing the s  disinfect t  unknown       -6.755  -0.002
✓U   For what reason does Tony Woodcock feel strongly ab  Johnson, wh  unknown       -2.915  -0.003
✓U   Which podcast discussed criticisms regarding Marcus  'Stick to F  unknown       -8.672  -0.000
✓U   What name did the comic character Lily Savage origi  

Saved → /kaggle/working/rome_results.json
{
  "config": {
    "model": "meta-llama/Llama-3.2-1B",
    "layer": 8
  },
  "counterfact": {
    "n_edits": 20,
    "metrics": {
      "efficacy": 0.95,
      "paraphrase": 0.875,
      "neighbourhood": 0.57,
      "score_HM": 0.7595177675535184
    }
  },
  "muse": {
    "n_forget": 10,
    "n_retain": 10,
    "metrics": {
      "forget_efficacy": 1.0,
      "forget_lp_flip": 1.0,
      "retain_locality": 0.3,
      "paraphrase": 0.0,
      "HM_FE_RL": 0.46153846282840244
    }
  }
}
